In [1]:
import pandas as pd
from database import postgres_connection


In [2]:
def load_data():
    conn = postgres_connection()

    try:
        user_df = pd.read_sql(
            """
            SELECT 
            * 
            FROM public.user_data
            """,
            conn,
        )

        post_df = pd.read_sql(
            """
            SELECT 
            *
            FROM public.post_text_df 
            """,
            conn,
        )

        feed_df = pd.read_sql(
            """
            SELECT 
            *
            FROM public.feed_data
            LIMIT 5000000
            """,
            conn,
        )

    except Exception as e:
        print(f"Ошибка запроса: {e}")
        raise

    finally:
        if conn:
            conn.close()

    return user_df, post_df, feed_df


# user_df, post_df, feed_df = load_data()

In [3]:
# print(user_df.shape)
# print(post_df.shape)
# print(feed_df.shape)

In [4]:
# print(user_df.head(5))
# print("=" * 100)
# print(post_df.head(5))
# print("=" * 100)
# print(feed_df.head(5))

In [5]:
# # Сохранение данных в data/ чтобы в случае перезапуска каждый раз не загружать
# import os

# os.makedirs("data", exist_ok=True)

# tabs_dict = {"user_df": user_df, "post_df": post_df, "feed_df": feed_df}

# for name, df in tabs_dict.items():
#     pathname = f"data/{name}_{df.shape[0]}rows"
#     df.to_csv(f"{pathname}.csv", index=False)
#     df.to_parquet(f"{pathname}.parquet", index=False)  # pip install pyarrow
#     print(f"Сохранено: {pathname}")


In [6]:
user_df = pd.read_csv("data/user_df_163205rows.csv", sep=",")
post_df = pd.read_csv("data/post_df_7023rows.csv", sep=",")
feed_df = pd.read_csv("data/feed_df_5000000rows.csv", sep=",")

print("Данные успешно загружены!")

Данные успешно загружены!


In [7]:
user_df_pqt = pd.read_parquet("data/user_df_163205rows.parquet")
post_df_pqt = pd.read_parquet("data/post_df_7023rows.parquet")
feed_df_pqt = pd.read_parquet("data/feed_df_5000000rows.parquet")

print("Данные успешно загружены!")

Данные успешно загружены!


In [8]:
print(user_df.shape)
print(post_df.shape)
print(feed_df.shape)

(163205, 8)
(7023, 3)
(5000000, 5)


In [9]:
print(user_df_pqt.shape)
print(post_df_pqt.shape)
print(feed_df_pqt.shape)

(163205, 8)
(7023, 3)
(5000000, 5)


In [10]:
user_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 163205 entries, 0 to 163204
Data columns (total 8 columns):
 #   Column     Non-Null Count   Dtype 
---  ------     --------------   ----- 
 0   user_id    163205 non-null  int64 
 1   gender     163205 non-null  int64 
 2   age        163205 non-null  int64 
 3   country    163205 non-null  object
 4   city       163205 non-null  object
 5   exp_group  163205 non-null  int64 
 6   os         163205 non-null  object
 7   source     163205 non-null  object
dtypes: int64(4), object(4)
memory usage: 10.0+ MB


In [11]:
user_df.isna().sum()

user_id      0
gender       0
age          0
country      0
city         0
exp_group    0
os           0
source       0
dtype: int64

In [12]:
user_df["user_id"].is_unique

True

In [13]:
post_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7023 entries, 0 to 7022
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   post_id  7023 non-null   int64 
 1   text     7023 non-null   object
 2   topic    7023 non-null   object
dtypes: int64(1), object(2)
memory usage: 164.7+ KB


In [14]:
post_df.isna().sum()

post_id    0
text       0
topic      0
dtype: int64

In [15]:
post_df["post_id"].is_unique

True

In [16]:
feed_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000000 entries, 0 to 4999999
Data columns (total 5 columns):
 #   Column     Dtype 
---  ------     ----- 
 0   timestamp  object
 1   user_id    int64 
 2   post_id    int64 
 3   action     object
 4   target     int64 
dtypes: int64(3), object(2)
memory usage: 190.7+ MB


In [17]:
feed_df.describe(include="all")

,timestamp,user_id,post_id,action,target
count,5000000,5.000000e+06,5.000000e+06,5000000,5.000000e+06
unique,1194999,NaN,NaN,2,NaN
top,2021-10-21 21:22:14,NaN,NaN,view,NaN
freq,36,NaN,NaN,4466001,NaN
mean,NaN,8.611754e+04,3.400139e+03,NaN,1.068000e-01
std,NaN,4.669992e+04,2.094610e+03,NaN,3.088588e-01
min,NaN,2.341000e+03,1.000000e+00,NaN,0.000000e+00
25%,NaN,5.385300e+04,1.529000e+03,NaN,0.000000e+00
50%,NaN,8.919400e+04,3.199000e+03,NaN,0.000000e+00
75%,NaN,1.236450e+05,5.208000e+03,NaN,0.000000e+00


In [18]:
feed_df.target.value_counts(dropna=False)

target
0    4466000
1     534000
Name: count, dtype: int64

In [19]:
feed_df.action.value_counts(dropna=False)

action
view    4466001
like     533999
Name: count, dtype: int64

In [20]:
feed_df.isna().sum()

timestamp    0
user_id      0
post_id      0
action       0
target       0
dtype: int64

In [21]:
pd.crosstab(
    feed_df["action"],
    feed_df["target"].fillna("missing"),
)

target,0,1
action,,
like,533999,0
view,3932001,534000


In [22]:
pd.crosstab(
    feed_df["action"],
    feed_df["target"].fillna("missing"),
    margins=True,
    normalize="index",
)

target,0,1
action,,
like,1.00000,0.00000
view,0.88043,0.11957
All,0.89320,0.10680


In [23]:
feed_df.target.isna().sum()

np.int64(0)

In [24]:
feed_df.loc[feed_df["action"] == "like", "target"].value_counts(dropna=False)

target
0    533999
Name: count, dtype: int64

In [25]:
feed_df[feed_df["action"] == "like"].head(5)

,timestamp,user_id,post_id,action,target
5,2021-12-07 06:48:44,15522,1638,like,0
9,2021-12-07 06:51:07,15522,3578,like,0
19,2021-12-09 20:58:03,15522,664,like,0
35,2021-12-09 21:23:14,15522,1049,like,0
46,2021-12-09 21:37:20,15522,1744,like,0


In [26]:
feed_df.groupby("action")[["user_id", "post_id"]].value_counts()

action  user_id  post_id
like    3219     3177       4
        137294   973        4
        2350     1655       3
        3384     6177       3
        15900    6503       3
                           ..
view    144107   7051       1
                 7121       1
                 7272       1
                 7280       1
                 7282       1
Name: count, Length: 4787657, dtype: int64

In [27]:
user_id, post_id = feed_df[feed_df["action"] == "like"][["user_id", "post_id"]].iloc[0]

feed_df[(feed_df["user_id"] == user_id) & (feed_df["post_id"] == post_id)]

,timestamp,user_id,post_id,action,target
4,2021-12-07 06:47:02,15522,1638,view,1
5,2021-12-07 06:48:44,15522,1638,like,0


В таблице feed_data событие лайка хранится отдельно от события просмотра. Для положительного взаимодействия сначала присутствует строка view с target=1, затем отдельная строка like с техническим значением target=0. Поэтому для формирования обучающей выборки используются только события view, а строки like исключаются, чтобы не создавать ложные отрицательные примеры.

In [28]:
view_df = feed_df[feed_df["action"] == "view"]
print(view_df.shape[0])
print(view_df.target.value_counts(dropna=False))
neg, pos = view_df.target.value_counts(dropna=False)
print(f"{pos / (neg + pos) * 100:.2f}%")

4466001
target
0    3932001
1     534000
Name: count, dtype: int64
11.96%


In [29]:
print(feed_df.user_id.nunique())
print(feed_df.post_id.nunique())
print(feed_df.timestamp.min())
print(feed_df.timestamp.max())

10588
6831
2021-10-01 06:01:40
2021-12-29 23:44:39


In [30]:
feed_df.groupby("user_id")["action"].count().sort_values(ascending=False).head(3)

user_id
116979    928
88153     904
89729     902
Name: action, dtype: int64

In [31]:
feed_df.head(5)

,timestamp,user_id,post_id,action,target
0,2021-12-07 06:40:46,15522,4012,view,0
1,2021-12-07 06:41:30,15522,1483,view,0
2,2021-12-07 06:42:53,15522,1829,view,0
3,2021-12-07 06:44:26,15522,1375,view,0
4,2021-12-07 06:47:02,15522,1638,view,1


# Создание признаков

In [32]:
print(user_df.head(5))
print("=" * 100)
print(post_df.head(5))
print("=" * 100)
print(feed_df.head(5))

   user_id  gender  age country               city  exp_group       os source
0      200       1   34  Russia          Degtyarsk          3  Android    ads
1      201       0   37  Russia             Abakan          0  Android    ads
2      202       1   17  Russia           Smolensk          4  Android    ads
3      203       0   18  Russia             Moscow          1      iOS    ads
4      204       0   36  Russia  Anzhero-Sudzhensk          3  Android    ads
   post_id                                               text     topic
0        1  UK economy facing major risks\n\nThe UK manufa...  business
1        2  Aids and climate top Davos agenda\n\nClimate c...  business
2        3  Asian quake hits European shares\n\nShares in ...  business
3        4  India power shares jump on debut\n\nShares in ...  business
4        5  Lacroix label bought by US firm\n\nLuxury good...  business
             timestamp  user_id  post_id action  target
0  2021-12-07 06:40:46    15522     4012   v

In [33]:
print(post_df.shape)
print(post_df.post_id.nunique())
print(post_df.text.nunique())

(7023, 3)
7023
6924


### Очистка и предобработка текстов постов


In [34]:
# import re
# import pandas as pd
# import spacy
# from sklearn.feature_extraction.text import TfidfVectorizer

# nlp = spacy.load("en_core_web_sm")

# # 1. создаем копию датафрейма с уникальными текстами постов
# # unique_texts_df = post_df.drop_duplicates(subset="text", keep="first")
# # print(unique_texts_df.shape)


# # 2. определяем функцию для предобработки текста
# def pre_clean(text):
#     if not isinstance(text, str):
#         return ""
#     text = text.lower()
#     text = re.sub(r"https?://\S+|www\.\S+", "", text)  # Удаляем ссылки
#     text = re.sub(r"[^a-z\s]", "", text)  # Оставляем только латиницу и пробелы

#     return text


# # применяем первичную очистку текста
# post_df["pre_cleaned_text"] = post_df["text"].apply(pre_clean)

# # 3. УСКОРЕННАЯ ЛЕММАТИЗАЦИЯ ЧЕРЕЗ nlp.pipe
# # Отключаем лишнее (ner, parser)
# cleaned_texts = []
# docs = nlp.pipe(
#     post_df["pre_cleaned_text"].astype("str"),
#     disable=["ner", "parser"],
# )

# for doc in docs:
#     # Убираем английские стоп-слова и берем лемму (начальную форму)
#     lemma = [token.lemma_ for token in doc if not token.is_stop and not token.is_punct]
#     cleaned_texts.append(" ".join(lemma))

# # Записываем результат в датафрейм
# post_df["cleaned_text"] = cleaned_texts


In [35]:
# # 4. СВЯЗКА С TF-IDF
# # max_features=1000 ограничит словарь до 50 самых важных слов, чтобы не перегружать память
# tfidf = TfidfVectorizer(max_features=50)

# # Обучаем TF-IDF на ускоренно очищенной колонке
# tfidf_matrix = tfidf.fit_transform(post_df["cleaned_text"])
# feature_names = tfidf.get_feature_names_out()
# # Переводим в DataFrame
# tfidf_df = pd.DataFrame(tfidf_matrix.toarray(), columns=feature_names).add_prefix(
#     "tfidf_"
# )
# print(tfidf_df.shape)

In [36]:
from text_processing import clean_and_vectorize_text

tfidf_processed_df = clean_and_vectorize_text(post_df["text"], max_features=10)

print(tfidf_processed_df.shape)
print(tfidf_processed_df.head(5))

(7023, 10)
   tfidf_film  tfidf_good  tfidf_like  tfidf_movie  tfidf_mr  tfidf_not  \
0         0.0    0.173100         0.0          0.0  0.000000        0.0   
1         0.0    0.000000         0.0          0.0  0.669338        0.0   
2         0.0    0.000000         0.0          0.0  0.000000        0.0   
3         0.0    0.425975         0.0          0.0  0.000000        0.0   
4         0.0    0.304438         0.0          0.0  0.419260        0.0   

   tfidf_people  tfidf_say  tfidf_time  tfidf_year  
0      0.000000   0.648420    0.173363    0.720786  
1      0.538564   0.455155    0.162255    0.168650  
2      0.721977   0.610162    0.326268    0.000000  
3      0.000000   0.797834    0.426621    0.000000  
4      0.000000   0.855300    0.000000    0.000000  


In [37]:
# Сбрасываем индексы исходного df, чтобы они шли строго от 0 до конца без пропусков
# unique_texts_df = unique_texts_df.reset_index(drop=True)
post_df.shape


(7023, 3)

In [38]:
# 5. Итоговый датасет: метаданные + векторные признаки
post_tfidf_processed_df = pd.concat(
    [post_df[["post_id", "topic"]], tfidf_processed_df], axis=1
)
print(post_tfidf_processed_df.shape)


(7023, 12)


In [39]:
post_tfidf_processed_df.head(5)

,post_id,topic,tfidf_film,tfidf_good,tfidf_like,tfidf_movie,tfidf_mr,tfidf_not,tfidf_people,tfidf_say,tfidf_time,tfidf_year
0,1,business,0.0,0.173100,0.0,0.0,0.000000,0.0,0.000000,0.648420,0.173363,0.720786
1,2,business,0.0,0.000000,0.0,0.0,0.669338,0.0,0.538564,0.455155,0.162255,0.168650
2,3,business,0.0,0.000000,0.0,0.0,0.000000,0.0,0.721977,0.610162,0.326268,0.000000
3,4,business,0.0,0.425975,0.0,0.0,0.000000,0.0,0.000000,0.797834,0.426621,0.000000
4,5,business,0.0,0.304438,0.0,0.0,0.419260,0.0,0.000000,0.855300,0.000000,0.000000


In [1]:
post_tfidf_processed_df.to_parquet(
    "data/post_features_7023rows.parquet",
    index=False,
)

NameError: name 'post_tfidf_processed_df' is not defined

### Оставляем только действия с action == view

In [40]:
feed_view_df = feed_df[feed_df["action"] == "view"]

print(feed_view_df.shape)
print(feed_view_df.head())


(4466001, 5)
             timestamp  user_id  post_id action  target
0  2021-12-07 06:40:46    15522     4012   view       0
1  2021-12-07 06:41:30    15522     1483   view       0
2  2021-12-07 06:42:53    15522     1829   view       0
3  2021-12-07 06:44:26    15522     1375   view       0
4  2021-12-07 06:47:02    15522     1638   view       1


### Добавляем временные признаки: hour, day_of_week, month

In [41]:
# from datetime import datetime as dt

# print(dt.now())

datetime_df = feed_view_df.timestamp  # .reset_index(drop=True)
print(datetime_df.head(5))
print(datetime_df.shape)
print(datetime_df.dtypes)


def hour_dow_mnth_from_timestamp(datetime_df):
    if not pd.api.types.is_datetime64_any_dtype(datetime_df):
        datetime_df = pd.to_datetime(datetime_df)
    temp_df = pd.DataFrame()
    temp_df["hour"] = datetime_df.dt.hour
    temp_df["dow"] = datetime_df.dt.weekday
    temp_df["month"] = datetime_df.dt.month

    return temp_df


date_df = hour_dow_mnth_from_timestamp(datetime_df)

print("=" * 100)
print(date_df.head(5))
print(date_df.shape)
print(date_df.dtypes)

0    2021-12-07 06:40:46
1    2021-12-07 06:41:30
2    2021-12-07 06:42:53
3    2021-12-07 06:44:26
4    2021-12-07 06:47:02
Name: timestamp, dtype: object
(4466001,)
object
   hour  dow  month
0     6    1     12
1     6    1     12
2     6    1     12
3     6    1     12
4     6    1     12
(4466001, 3)
hour     int32
dow      int32
month    int32
dtype: object


In [42]:
print(feed_view_df.shape)
print(date_df.shape)

feed_view_df = pd.concat([feed_view_df, date_df], axis=1)

(4466001, 5)
(4466001, 3)


In [43]:
feed_view_df.head()

,timestamp,user_id,post_id,action,target,hour,dow,month
0,2021-12-07 06:40:46,15522,4012,view,0,6,1,12
1,2021-12-07 06:41:30,15522,1483,view,0,6,1,12
2,2021-12-07 06:42:53,15522,1829,view,0,6,1,12
3,2021-12-07 06:44:26,15522,1375,view,0,6,1,12
4,2021-12-07 06:47:02,15522,1638,view,1,6,1,12


In [44]:
learning_table = pd.merge(feed_view_df, user_df, on="user_id", how="left")
learning_table = learning_table.drop("action", axis=1)
print(learning_table.shape)
learning_table.head()

(4466001, 14)


,timestamp,user_id,post_id,target,hour,dow,month,gender,age,country,city,exp_group,os,source
0,2021-12-07 06:40:46,15522,4012,0,6,1,12,1,34,Russia,Krasnoyarsk,4,Android,ads
1,2021-12-07 06:41:30,15522,1483,0,6,1,12,1,34,Russia,Krasnoyarsk,4,Android,ads
2,2021-12-07 06:42:53,15522,1829,0,6,1,12,1,34,Russia,Krasnoyarsk,4,Android,ads
3,2021-12-07 06:44:26,15522,1375,0,6,1,12,1,34,Russia,Krasnoyarsk,4,Android,ads
4,2021-12-07 06:47:02,15522,1638,1,6,1,12,1,34,Russia,Krasnoyarsk,4,Android,ads


In [45]:
final_table = pd.merge(
    learning_table, post_tfidf_processed_df, on="post_id", how="left"
)
print(final_table.shape)
final_table.head()

(4466001, 25)


,timestamp,user_id,post_id,target,hour,dow,month,gender,age,country,...,tfidf_film,tfidf_good,tfidf_like,tfidf_movie,tfidf_mr,tfidf_not,tfidf_people,tfidf_say,tfidf_time,tfidf_year
0,2021-12-07 06:40:46,15522,4012,0,6,1,12,1,34,Russia,...,0.0,0.000000,0.000000,0.0,0.000000,0.0,0.000000,0.000000,0.000000,0.000000
1,2021-12-07 06:41:30,15522,1483,0,6,1,12,1,34,Russia,...,0.0,0.000000,0.318844,0.0,0.000000,0.0,0.000000,0.000000,0.947807,0.000000
2,2021-12-07 06:42:53,15522,1829,0,6,1,12,1,34,Russia,...,0.0,0.000000,0.000000,0.0,0.000000,0.0,0.000000,0.889328,0.317030,0.329527
3,2021-12-07 06:44:26,15522,1375,0,6,1,12,1,34,Russia,...,0.0,0.397306,0.000000,0.0,0.273577,0.0,0.220127,0.186035,0.000000,0.827187
4,2021-12-07 06:47:02,15522,1638,1,6,1,12,1,34,Russia,...,0.0,0.419034,0.000000,0.0,0.000000,0.0,0.000000,0.000000,0.629506,0.654319


In [46]:
final_table.isna().sum()[final_table.isna().sum() > 0]
# final_table.loc[:, final_table.isna().sum().any()]
final_table.isna().sum().any()

np.False_

In [47]:
# Проверка итоговой таблицы на пропуски и отсутствующие значения

missing_counts = final_table.isna().sum().sort_values(ascending=False)
missing_share = final_table.isna().mean().sort_values(ascending=False)

print("Пропуски по столбцам:")
print(missing_counts[missing_counts > 0])

print("\nДоля пропусков по столбцам:")
print(missing_share[missing_share > 0])

rows_with_missing = final_table.isna().any(axis=1).sum()
print(f"\nСтрок с хотя бы одним пропуском: {rows_with_missing}")
print(f"Доля строк с пропусками: {rows_with_missing / len(final_table):.2%}")

# Посмотреть первые строки, где есть пропуски
if rows_with_missing > 0:
    final_table[final_table.isna().any(axis=1)].head()
else:
    print("\nПропусков в final_table не найдено.")

Пропуски по столбцам:
Series([], dtype: int64)

Доля пропусков по столбцам:
Series([], dtype: float64)

Строк с хотя бы одним пропуском: 0
Доля строк с пропусками: 0.00%

Пропусков в final_table не найдено.


In [48]:
print(
    f"{final_table.memory_usage(deep=True).sum() / (1024**2):.2f} Mbites"
)  # в мегабайтах

1894.58 Mbites


In [53]:
print(final_table.timestamp.min())
print(final_table.timestamp.max())

2021-10-01 06:01:40
2021-12-29 23:44:39


In [64]:
# количество взаимодействий;

# долю положительного класса;
# число уникальных пользователей;
# число уникальных постов.

# В результате должна получиться компактная сводка с одной строкой на месяц.
monthly_stats = final_table.groupby("month").agg(
    interactions=("target", "size"),
    positive_rate=("target", lambda x: round(x.mean() * 100, 2)),
    unique_users=("user_id", "nunique"),
    unique_posts=("post_id", "nunique"),
)
print(monthly_stats)

       interactions  positive_rate  unique_users  unique_posts
month                                                         
10          1517741           9.39         10458          6831
11          1485926          12.82         10450          6831
12          1462334          13.74         10373          6831


После уменьшения числа TF-IDF-признаков до 10 итоговая обучающая таблица содержит 4 466 001 строку и 25 столбцов. Объём памяти снизился примерно до 1.85 ГБ. После объединения пользовательских, постовых и временных признаков пропуски отсутствуют. Для оценки модели планируется временное разделение: данные октября–ноября используются для обучения, данные декабря — для тестирования.

In [58]:
# Теперь нужно проверить холодный старт:

# есть ли в декабре пользователи, которых нет в октябре–ноябре;
users10_11 = final_table[final_table["month"].isin([10, 11])]["user_id"].unique()
users12 = final_table[final_table["month"] == 12]["user_id"].unique()

# есть ли в декабре посты, которых нет в октябре–ноябре.

posts10_11 = final_table[final_table["month"].isin([10, 11])]["post_id"].unique()
posts12 = final_table[final_table["month"] == 12]["post_id"].unique()

new_users_in_december = set(users12) - set(users10_11)
new_posts_in_december = set(posts12) - set(posts10_11)

print(
    f"Количество новых пользователей в декабре (холодный старт): {len(new_users_in_december)}"
)
print(
    f"Количество новых постов в декабре (холодный старт): {len(new_posts_in_december)}"
)


Количество новых пользователей в декабре (холодный старт): 1
Количество новых постов в декабре (холодный старт): 0


In [60]:
train_df = final_table[final_table["month"].isin([10, 11])]
test_df = final_table[final_table["month"] == 12]
print(train_df.shape)
print(test_df.shape)

train_users = set(train_df["user_id"])
test_users = set(test_df["user_id"])
new_users = test_users - train_users
print(f"Количество новых пользователей в декабре (холодный старт): {len(new_users)}")

train_posts = set(train_df["post_id"])
test_posts = set(test_df["post_id"])
new_posts = test_posts - train_posts
print(f"Количество новых постов в декабре (холодный старт): {len(new_posts)}")


(3003667, 25)
(1462334, 25)
Количество новых пользователей в декабре (холодный старт): 1
Количество новых постов в декабре (холодный старт): 0


Для оценки качества использовано временное разделение: октябрь–ноябрь — обучающая выборка, декабрь — тестовая. Проверка холодного старта показала, что в тестовой выборке присутствует только один новый пользователь и отсутствуют новые посты. Поэтому выбранное разделение позволяет оценить качество модели на более позднем периоде без значительной проблемы неизвестных объектов.

In [ ]:
# Итоговая таблица для обучения модели готова и сохраняется в формате parquet для экономии места и ускорения загрузки:
final_table.to_parquet("data/final_table.parquet", index=False)

In [62]:
df = pd.read_parquet("data/final_table.parquet")

In [63]:
print(df.shape)
print(df.memory_usage(deep=True).sum() / (1024**2))  # в мегабайтах
print(df.isna().sum().sum())  # количество пропусков в таблице

(4466001, 25)
1894.5805072784424
0


In [64]:
print(final_table.shape == df.shape)
print(final_table.columns.equals(df.columns))

True
True
